# 跑步数据探索

这个 notebook 只调用分析包 `running` 的公开函数，和仪表盘用的是同一套计算，所以这里看到的数字和仪表盘一致。

数据目录由环境变量 `RUNNING_DATA_DIR` 决定，默认 `%USERPROFILE%\running-data`。先运行 `runsync sync` 拿到新数据。

输出里有个人数据：提交时 `nbstripout` 会清掉输出。想保存自己的探索，复制一份到仓库外面。

In [ ]:
import polars as pl

from running.aerobic import aerobic_runs, aerobic_trend, load_records, steady_state
from running.config import data_dir, load_config
from running.db import open_analysis_db
from running.load import training_load
from running.runwalk import empty_run_summaries, running_summaries
from running.status import data_status
from running.volume import weekly_volume

print("数据目录:", data_dir())
config = load_config()
db = open_analysis_db()
data_status()

## 周跑量

周一开始的自然周；跑步配速只算“跑”的秒，走路不拖慢配速。

In [ ]:
volume = weekly_volume()
print("本周:", volume["this_week"])
print("最近 7 天:", volume["rolling_7d"])
pl.DataFrame(volume["weeks"])

## 训练负荷

Banister TRIMP；CTL（42 天）、ATL（7 天）、TSB = 前一天的 CTL − ATL。

In [ ]:
load = training_load()
print("当前:", load["current"])
print("负荷突增:", load["spike"], load["spike_since"])
print("缺心率的活动:", load["missing_hr_count"])
pl.DataFrame(load["series"]).tail(28)

## 有氧能力

每次户外路跑在参考心率处的 GAP 速度，按周取中位数，再取 4 周滚动中位数。`range` 可选 `6m`、`1y`、`2y`、`all`。

In [ ]:
trend = aerobic_trend("1y")
pl.DataFrame(trend["weeks"]).filter(pl.col("runs") > 0) if trend["weeks"] else pl.DataFrame()

## 深入：每次跑步

下面直接拿 polars 表，可以自己筛选、分组、对比。

In [ ]:
runs = running_summaries(db, config.run_walk) if db else empty_run_summaries()
runs.sort("start_time_local", descending=True).head(20)

In [ ]:
road = aerobic_runs(db, config) if db else pl.DataFrame()
road.sort("start_time_local", descending=True).head(20) if db else road

## 深入：单次跑步的逐秒数据

取最近一次路跑，看哪些秒被算作稳态段，以及稳态段上的 EF（GAP 速度 ÷ 心率）。改 `activity_id` 看别的跑步。

In [ ]:
if db is not None and road.height > 0:
    activity_id = road.sort("start_time_local")["activity_id"][-1]
    seconds = steady_state(load_records(db, [activity_id]), config.aerobic, config.run_walk)
    steady_rows = seconds.filter(pl.col("steady"))
    print("活动:", activity_id, "稳态秒数:", steady_rows["dt_s"].sum())
    if steady_rows.height > 0:
        print("EF:", steady_rows["gap_mps"].mean() / steady_rows["hr_smooth"].mean())
else:
    seconds = pl.DataFrame()
    print("还没有路跑数据")
seconds.head(10)

In [ ]:
if db is not None:
    db.close()